# Aperture correction for a point source

In an integral-field data cube, a small spatial aperture captures only part of a point source's light in each wavelength channel. Because the PSF can vary with wavelength, this fraction can also change across a spectrum. A normalized PSF lets us estimate the missing light and recover an integrated flux density under the point-source assumption.

All three approaches below require a PSF model applicable to the observations. Here it is calibrated using an isolated field star with the same spatial sampling and wavelength convention. [Chromatic PSF](Chromatic%20PSF.ipynb) shows that calibration; a compact model is supplied so this example can run independently.

| Estimate | Operation | When useful / limitation |
| --- | --- | --- |
| `aperture` | Divide the aperture sum by the wavelength-dependent PSF fraction. | Corrects chromatic aperture losses; any extended emission inside the aperture is rescaled too. |
| Constant correction | Multiply the aperture sum by the mean PSF-derived correction. | Avoids transferring detailed chromatic fluctuations in the correction, but loses real chromatic variation. Still requires the PSF model. |
| `optimal` | Fit a fixed PSF amplitude and a constant background at each wavelength. | Minimum-variance linear estimate with a correct spatial model and known independent variances; unknown spatial structure can bias it. |

None of these methods guarantees separation of a point source from arbitrary surrounding emission.

## Workflow at a glance

**Inputs:** a flux-density cube with a variance cube, a normalized wavelength-dependent PSF calibrated from an isolated point source, and aperture/fit radii in pixels. The PSF must match the spatial sampling and cover the wavelengths to extract.

Start with [Chromatic PSF](Chromatic%20PSF.ipynb) to calibrate a model from an isolated star, or use the supplied model to follow the extraction directly.

1. Load the cube and PSF; estimate a fixed source center from the mean image.
2. Define the small extraction aperture and the larger PSF-fitting region.
3. **Method 1:** extract, inspect and save the wavelength-dependent aperture correction.
4. **Method 2:** rescale the raw aperture spectrum from Method 1 by its mean correction.
5. **Method 3:** independently fit, inspect and save the PSF-plus-constant spectrum.
6. Compare the three spectra, statistical errors and correction factors.

**Outputs:** three spectra, diagnostic tables and a three-panel comparison. FITS files preserve units and errors; the comparison is also saved as a PDF. Change the paths and radii below for another source.

In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling.models import Const2D
from IPython.display import display

from prism.data import Cube
from prism.utils.plotting import plot_outline
from prism.psf import fitimage, ChromaticPSF, aperture, optimal
from prism.modeling.models import MoffatPSF
from prism.psf.extraction import save, read

In [ ]:
cubepath = Path('data/pointsource.fits')
psfpath = Path('data/psf.fits')
outpath = Path('results/extraction')
outpath.mkdir(parents=True, exist_ok=True)

# Load the calibrated correction PSF and the observed cube.
psf = ChromaticPSF.from_fits(psfpath)
cube = Cube.from_fits(cubepath, ext_values='DATA', ext_var='VAR')
img = cube.get_image(method='mean')
cy, cx = np.unravel_index(np.nanargmax(img.values), img.shape)
y, x = np.indices(img.shape)

## Fixed target center

Fit a Prism `MoffatPSF` with FWHM and wing exponent `beta` plus a constant to the mean image, using only the central region. This fit estimates a center to hold fixed during extraction; its shape is not used for aperture correction. The correction still uses the independent chromatic PSF calibrated on the field star.

The commented alternative fixes the calibrated PSF shape and fits only amplitude and centroid. All radii and coordinates below are in local pixels.

In [ ]:
# Define a simple profile for centering; this is not the correction PSF.
model = MoffatPSF(
    amplitude=float(img.values[cy, cx] - np.nanmedian(img.values)),
    x_0=float(cx), y_0=float(cy), fwhm=3, beta=2.5,
    bounds={'fwhm': (0.3, 12), 'beta': (1.05, 12)},
)

# Alternative: reuse the calibrated shape, fitting only amplitude and centroid.
# model = psf(float(np.mean(psf.domain)))
# for name in model.param_names:
#     model.fixed[name] = name not in ('amplitude', 'x_0', 'y_0')
# model.x_0, model.y_0 = float(cx), float(cy)
# model.amplitude = float(img.values[cy, cx] - np.nanmedian(img.values))

model.x_0.bounds = (cx-2, cx+2)
model.y_0.bounds = (cy-2, cy+2)

In [ ]:
# Fit the emission centroid in the central region.
initial = fitimage(img, model, mask=((x-cx)**2 + (y-cy)**2 <= 4**2),
                   background=Const2D(float(np.nanmedian(img.values))), factor=psf.factor)
center = (float(initial['y_0']), float(initial['x_0']))

Inspect `success` and the fit diagnostics before adopting the fixed center.

In [ ]:
print('Fixed center (y, x):', center)
display(initial)

## Extraction regions

Define both pixel-center selections once: the small aperture for Methods 1 and 2, and the larger fit region for Method 3. Outlines follow the exact selected pixel edges.

In [ ]:
radius = 1.0
fitradius = 3.0
mask = (x-center[1])**2 + (y-center[0])**2 <= radius**2
fitmask = (x-center[1])**2 + (y-center[0])**2 <= fitradius**2
# Binary masks select pixels by their centers; these are not fractional circles.

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(np.arcsinh(img.values), origin='lower', cmap='viridis')
plot_outline(mask, ax=ax, color='cyan')
plot_outline(fitmask, ax=ax, color='orange')
ax.scatter(center[1], center[0], marker='x', color='red')
ax.set(xlabel='x [pixel]', ylabel='y [pixel]', title='Aperture (cyan), PSF fit (orange)')
plt.show()

## Method 1 — Wavelength-dependent aperture correction

Sum the small aperture and divide by the fraction of the normalized PSF captured by the same mask. The PSF is normalized on the infinite plane, so the finite-aperture fraction is not one. This retains any surrounding emission inside the aperture and rescales it too; prefer this composite spectrum when the surrounding morphology is unknown.

In [ ]:
# Correct the aperture sum for wavelength-dependent PSF losses.
spectrum, diagnostics = aperture(cube, psf, mask, center=center)

In [ ]:
print(f"Aperture: {np.count_nonzero(diagnostics['success'])}/{len(diagnostics)} valid channels")
print('Aperture pixels:', mask.sum())
display(diagnostics[['wave', 'fraction', 'success']][::200])

In [ ]:
# Save the spectrum and its diagnostics.
diagnostics.meta['source'] = str(cubepath)
diagnostics.meta['psf_source'] = str(psfpath)
save(diagnostics, outpath / 'aperture-spectrum.fits', overwrite=True)

## Method 2 — Constant aperture correction

Take the unweighted mean of the valid wavelength-dependent aperture corrections and apply it to the raw aperture spectrum. This factor still comes from the PSF model. It smooths away both spurious and real chromatic variations; it does not remove surrounding emission. The wavelength-dependent spread of the corrections is not, by itself, an uncertainty estimate.

This comparison reuses the raw aperture sum and PSF fractions from Method 1.

In [ ]:
# Apply the mean correction to the same raw aperture sum and error.
correction = 1 / np.asarray(diagnostics['fraction'])
valid = np.asarray(diagnostics['success']) & np.isfinite(correction)
scale = float(np.mean(correction[valid]))
constant = diagnostics[['wave', 'rawflux', 'rawerr', 'success']].copy()
constant.rename_columns(['rawflux', 'rawerr'], ['flux', 'flux_err'])
constant['flux'] *= scale
constant['flux_err'] *= scale

In [ ]:
print(f'Mean aperture correction: {scale:.4f}')

In [ ]:
# Save the constant-correction spectrum.
constant.meta['method'] = 'constant aperture correction'
constant.meta['correction'] = scale
constant.meta['averaging'] = 'Unweighted mean of 1/fraction over valid channels'
save(constant, outpath / 'constant-spectrum.fits', overwrite=True)

## Method 3 — PSF + constant background (`optimal`)

Fit the calibrated PSF and a spatially constant background to each wavelength image. This estimates the integrated point-source amplitude directly, without a second aperture correction. Use a region large enough to distinguish the PSF wings from the background.

The estimate is optimal in the minimum-variance linear sense when the spatial model is correct and the input variances describe independent noise. Unknown compact or structured emission can bias the fitted amplitude; use the aperture methods when the desired spectrum should retain such emission.

PSF-based optimal extraction is also described in the [STScI JWST pipeline documentation](https://jwst-pipeline.readthedocs.io/en/stable/jwst/extract_1d/description.html). Here the same general profile-fitting principle is applied to cube images with a fixed calibrated PSF and a constant background.

In [ ]:
# Fit the calibrated PSF amplitude and a constant in each channel.
fitted, fitinfo = optimal(cube, psf, fitmask, center=center, background=Const2D())

In [ ]:
print(f"PSF fit: {np.count_nonzero(fitinfo['success'])}/{len(fitinfo)} valid channels")
print('PSF-fit pixels:', fitmask.sum())
display(fitinfo[['wave', 'background', 'redchi2', 'success']][::200])

In [ ]:
# Save the fitted spectrum and its diagnostics.
fitinfo.meta['source'] = str(cubepath)
fitinfo.meta['psf_source'] = str(psfpath)
save(fitinfo, outpath / 'optimal-spectrum.fits', overwrite=True)

## Compare spectra, errors and correction factors

Compare the spectral shapes and line fluxes, then inspect how strongly each correction changes the raw aperture spectrum. In the lower panel, the PSF-fit/raw-aperture ratio includes the effect of background subtraction and different spatial weighting; it is a diagnostic comparison, not a separate PSF aperture correction.

The error curves show propagated one-sigma measurement errors with the adopted PSF and center held fixed. They do not include PSF-calibration uncertainty or correlations introduced by cube resampling.

In [ ]:
raw = np.asarray(diagnostics['rawflux'])
ratio = np.full(raw.shape, np.nan)
valid = np.asarray(fitinfo['success']) & np.asarray(diagnostics['success']) & np.isfinite(raw) & (raw != 0)
np.divide(fitted.y, raw, out=ratio, where=valid)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True, constrained_layout=True)
axes[0].plot(spectrum.x, spectrum.y, label='Aperture corrected', color='black', lw=.8)
axes[0].plot(fitted.x, fitted.y, label='PSF + constant', color='tab:orange', lw=.8)
axes[0].plot(spectrum.x, constant['flux'], label='Constant correction', color='tab:blue', lw=.8)
axes[0].set_ylabel(f'Flux [{cube.unit}]')
axes[0].legend()
axes[1].plot(spectrum.x, spectrum.yerr, color='black', label='Aperture error')
axes[1].plot(fitted.x, fitted.yerr, color='tab:orange', label='PSF-fit error')
axes[1].plot(spectrum.x, constant['flux_err'], color='tab:blue', label='Constant-correction error')
axes[1].set_ylabel('Conditional statistical error')
axes[1].legend()

axes[2].plot(spectrum.x, correction, color='black', label='PSF aperture correction')
axes[2].plot(spectrum.x, ratio, color='tab:orange', lw=.8, label='PSF-fit flux / raw aperture flux (diagnostic)')
axes[2].axhline(scale, color='tab:blue', label='Mean aperture correction')
axes[2].set(xlabel=f'Wavelength [{cube.zunit}]', ylabel='Flux / raw aperture flux')
axes[2].legend()
fig.savefig(outpath / 'extraction-comparison.pdf')
plt.show()

Differences between the spectra can reflect spatial weighting, background assumptions or PSF mismatch. Repeat with different aperture and fit radii to assess sensitivity; this is a robustness check, not a calibrated uncertainty. The plotted errors alone do not quantify these effects. Physical interpretations, including luminosity estimates, must account for any surrounding emission retained in the chosen spectrum.

## Save and reuse the spectra

Each method saves a FITS file ready for further spectral analysis. The `SPECTRUM` extension contains wavelength, flux density and one-sigma error, with units. Numerical diagnostics and the extraction aperture are stored alongside it so the measurement can be inspected later.

Use `read` to recover the spectrum and its diagnostics, or `Spectrum.from_fits(..., ext='SPECTRUM')` to load the spectrum alone. The example below reloads the spectrum and its diagnostics.

In [ ]:
saved_spectrum, saved_diagnostics = read(outpath / "aperture-spectrum.fits")
saved_diagnostics[["wave", "flux", "flux_err", "fraction"]][::200]